# 02 — Snapshot Feature Engineering for Dropout Classification

## Purpose
Create leakage-aware, student-level datasets for conventional binary classification models such as Random Forest, Gradient Boosting, and XGBoost. **This notebook performs data preparation and feature creation only; model fitting belongs in the next notebook/stage.**

### Prediction designs prepared
1. **Primary:** one row per student at a fixed 14-day-after-signup cutoff.
2. **Sensitivity:** 7-day and 30-day signup snapshots, saved separately/with a snapshot identifier.
3. **Attempt-count alternative:** features from the first three attempts after signup, for students who have at least three eligible attempts. This is an exploratory comparison only because it excludes low-activity/no-attempt students and therefore changes the population.

### Target limitation
`outcome.csv` contains a student-level `dropped_out` label but no dropout timestamp. We cannot verify whether the outcome occurred after any chosen cutoff. These are therefore **snapshot-feature datasets with an assumed future target**, not proof of a temporally validated early-warning target. State this limitation in every evaluation.

### Feature exclusions
- `refund_requested`: excluded from model features until its timestamp and meaning are verified; it may be a post-decision/leakage signal.
- `explanations.csv`: excluded from temporal features because it has no submission timestamp.
- IDs: retained as identifiers for joins/audits, never as model features.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

ROOT_CANDIDATES = [Path("."), Path(".."), Path("/mnt/data/student_success_assistant_v2")]
ROOT = next((p.resolve() for p in ROOT_CANDIDATES if (p / "data" / "students.csv").exists()), Path(".").resolve())
DATA = ROOT / "data"
CURATED = DATA / "curated"
CURATED.mkdir(parents=True, exist_ok=True)

students_raw = pd.read_csv(DATA / "students.csv")
attempts_raw = pd.read_csv(DATA / "quiz_attempts.csv")
outcomes_raw = pd.read_csv(DATA / "outcome.csv")
explanations_raw = pd.read_csv(DATA / "explanations.csv")

print("Project root:", ROOT)
for name, df in [("students", students_raw), ("quiz_attempts", attempts_raw),
                 ("outcome", outcomes_raw), ("explanations", explanations_raw)]:
    print(f"{name:16s}: {df.shape[0]:,} rows x {df.shape[1]} columns")
display(students_raw.head())
display(attempts_raw.head())


Project root: /Users/abishek/Documents/Projects/student_success_assistant
students        : 500 rows x 6 columns
quiz_attempts   : 2,453 rows x 6 columns
outcome         : 500 rows x 2 columns
explanations    : 724 rows x 5 columns


,student_id,age_group,country,signup_date,device,refund_requested
0,S001,35-44,UAE,2026-07-18,mobile,no
1,S002,18-24,Sri Lanka,2026-07-11,mobile,no
2,S003,35-44,Sri Lanka,2026-07-27,mobile,no
3,S004,25-34,India,2026-07-29,mobile,no
4,S005,18-24,India,2026-07-13,desktop,no


,attempt_id,student_id,lesson_id,score,time_spent_min,attempted_at
0,A0001,S437,L1,99,22.7,2026-07-05 23:34
1,A0002,S139,L2,75,23.2,2026-07-21 12:03
2,A0003,S235,L1,91,36.3,2026-07-06 23:18
3,A0004,S496,L2,85,18.6,2026-08-05 14:25
4,A0005,S332,L2,84,22.5,2026-09-04 02:24


## 1. Normalize and validate the inputs

In [2]:
students = students_raw.copy()
attempts = attempts_raw.copy()
outcomes = outcomes_raw.copy()

# Parse dates without discarding original input files.
students["signup_dt"] = pd.to_datetime(students["signup_date"], errors="coerce")
attempts["attempt_dt"] = pd.to_datetime(attempts["attempted_at"], errors="coerce")
attempts["score_num"] = pd.to_numeric(attempts["score"], errors="coerce")
attempts["duration_num"] = pd.to_numeric(attempts["time_spent_min"], errors="coerce")

# Standardize known categorical strings without silently inventing new categories.
for col in ["age_group", "country", "device"]:
    students[col] = students[col].astype("string").str.strip()
    students.loc[students[col].eq(""), col] = pd.NA

students["refund_clean_audit_only"] = (
    students["refund_requested"].astype("string").str.strip().str.lower()
)
outcomes["dropped_out"] = outcomes["dropped_out"].astype("string").str.strip().str.lower()
outcomes["target_dropout"] = outcomes["dropped_out"].map({"yes": 1, "no": 0})

# Fail loudly on ambiguous join keys instead of duplicating rows downstream.
assert students["student_id"].notna().all(), "Missing student_id in students"
assert students["student_id"].is_unique, "students.student_id must be unique"
assert outcomes["student_id"].notna().all() and outcomes["student_id"].is_unique, "Outcome IDs must be unique/non-null"
assert set(outcomes["student_id"]).issubset(set(students["student_id"])), "Outcome contains unknown student IDs"
assert outcomes["target_dropout"].notna().all(), "Unexpected/missing outcome labels require investigation"
assert attempts["attempt_id"].notna().all() and attempts["attempt_id"].is_unique, "Attempt IDs must be unique/non-null"
assert set(attempts["student_id"].dropna()).issubset(set(students["student_id"])), "Attempt references unknown student"

# Valid score is a scoring property, not a reason to delete the whole attempt.
attempts["score_valid"] = attempts["score_num"].between(0, 100, inclusive="both")
attempts["duration_valid"] = attempts["duration_num"].ge(0)
attempts["score_invalid_flag"] = (~attempts["score_valid"]).astype("int8")
attempts["duration_missing_flag"] = attempts["duration_num"].isna().astype("int8")
attempts["duration_negative_flag"] = (attempts["duration_num"] < 0).fillna(False).astype("int8")

# Join signup date only after checking unique student dimension.
attempts = attempts.merge(students[["student_id", "signup_dt"]], on="student_id", how="left", validate="many_to_one")
attempts["days_from_signup"] = (
    attempts["attempt_dt"].dt.normalize() - attempts["signup_dt"].dt.normalize()
).dt.days

quality_summary = pd.DataFrame([
    {"check": "students", "rows": len(students), "issues": int(students["signup_dt"].isna().sum()), "issue_definition": "unparseable signup date"},
    {"check": "attempts", "rows": len(attempts), "issues": int(attempts["attempt_dt"].isna().sum()), "issue_definition": "unparseable attempt timestamp"},
    {"check": "attempts", "rows": len(attempts), "issues": int((attempts["days_from_signup"] < 0).sum()), "issue_definition": "attempt before signup"},
    {"check": "attempts", "rows": len(attempts), "issues": int((~attempts["score_valid"]).sum()), "issue_definition": "missing/non-numeric/out-of-range score"},
    {"check": "attempts", "rows": len(attempts), "issues": int((attempts["duration_num"] < 0).sum()), "issue_definition": "negative duration"},
])
display(quality_summary)
quality_summary.to_csv(CURATED / "snapshot_input_quality_checks.csv", index=False)


,check,rows,issues,issue_definition
0,students,500,0,unparseable signup date
1,attempts,2453,0,unparseable attempt timestamp
2,attempts,2453,0,attempt before signup
3,attempts,2453,3,missing/non-numeric/out-of-range score
4,attempts,2453,0,negative duration


## 2. Feature functions and explicit rules

In [3]:
HALF_LIFE_DAYS = 14

def weighted_score(scores, timestamps, cutoff, half_life_days=HALF_LIFE_DAYS):
    """Recency-weighted score. All timestamps must be <= the snapshot cutoff."""
    mask = scores.notna() & timestamps.notna() & (timestamps <= cutoff)
    s = scores.loc[mask].astype(float)
    t = timestamps.loc[mask]
    if len(s) == 0:
        return np.nan
    age_days = (cutoff - t).dt.total_seconds().to_numpy() / 86400.0
    weights = np.exp(-np.log(2) * age_days / half_life_days)
    return float(np.average(s.to_numpy(), weights=weights))

def score_slope_days(scores, timestamps):
    """OLS slope of valid score against elapsed days; NaN if fewer than 2 distinct timestamps."""
    ok = scores.notna() & timestamps.notna()
    if int(ok.sum()) < 2:
        return np.nan
    t = timestamps.loc[ok]
    x = (t - t.min()).dt.total_seconds().to_numpy() / 86400.0
    y = scores.loc[ok].astype(float).to_numpy()
    if np.unique(x).size < 2:
        return np.nan
    return float(np.polyfit(x, y, 1)[0])

def build_snapshot_row(student_row, eligible_attempts, cutoff, snapshot_days, target):
    """Build one feature row for one student and one fixed signup-relative cutoff."""
    d = eligible_attempts.copy()
    # Events must be after signup and at/before cutoff; bad/missing event times are ineligible.
    d = d[
        d["attempt_dt"].notna()
        & d["signup_dt"].notna()
        & (d["attempt_dt"] >= d["signup_dt"])
        & (d["attempt_dt"] <= cutoff)
    ].sort_values(["attempt_dt", "attempt_id"])

    valid = d[d["score_valid"] & d["score_num"].notna()].copy()
    durations = d.loc[d["duration_valid"] & d["duration_num"].notna(), "duration_num"]
    row = {
        "student_id": student_row["student_id"],
        "snapshot_days": int(snapshot_days),
        "snapshot_cutoff": cutoff,
        "target_dropout": int(target),
        # Student attributes allowed as candidate predictors; no ID/refund/explanation.
        "age_group": student_row.get("age_group", pd.NA),
        "country": student_row.get("country", pd.NA),
        "device": student_row.get("device", pd.NA),
        "signup_weekday": int(student_row["signup_dt"].dayofweek) if pd.notna(student_row["signup_dt"]) else np.nan,
        "signup_month": int(student_row["signup_dt"].month) if pd.notna(student_row["signup_dt"]) else np.nan,
        "attempt_count": int(len(d)),
        "valid_score_count": int(len(valid)),
        "invalid_score_count": int((~d["score_valid"]).sum()),
        "score_missing_count": int(d["score_num"].isna().sum()),
        "distinct_lessons_attempted": int(d["lesson_id"].nunique(dropna=True)),
        "active_days": int(d["attempt_dt"].dt.normalize().nunique()),
        "mean_score": float(valid["score_num"].mean()) if len(valid) else np.nan,
        "median_score": float(valid["score_num"].median()) if len(valid) else np.nan,
        "std_score": float(valid["score_num"].std(ddof=1)) if len(valid) > 1 else np.nan,
        "min_score": float(valid["score_num"].min()) if len(valid) else np.nan,
        "max_score": float(valid["score_num"].max()) if len(valid) else np.nan,
        "latest_valid_score": float(valid.sort_values("attempt_dt")["score_num"].iloc[-1]) if len(valid) else np.nan,
        "recency_weighted_score_14d_half_life": weighted_score(valid["score_num"].reset_index(drop=True), valid["attempt_dt"].reset_index(drop=True), cutoff),
        "score_trend_points_per_day": score_slope_days(valid["score_num"].reset_index(drop=True), valid["attempt_dt"].reset_index(drop=True)),
        "mean_duration_min": float(durations.mean()) if len(durations) else np.nan,
        "median_duration_min": float(durations.median()) if len(durations) else np.nan,
        "duration_observed_count": int(len(durations)),
        "duration_missing_count": int(d["duration_num"].isna().sum()),
        "duration_negative_count": int((d["duration_num"] < 0).sum()),
        "no_attempts_flag": int(len(d) == 0),
        "no_valid_scores_flag": int(len(valid) == 0),
        "days_to_first_attempt": float((d["attempt_dt"].min() - student_row["signup_dt"]).total_seconds() / 86400) if len(d) and pd.notna(student_row["signup_dt"]) else np.nan,
        "days_since_last_attempt_at_cutoff": float((cutoff - d["attempt_dt"].max()).total_seconds() / 86400) if len(d) else np.nan,
    }
    # Lesson-specific score summaries avoid assuming lessons are equally difficult.
    for lesson in ["L1", "L2", "L3"]:
        dl = valid[valid["lesson_id"].astype("string").eq(lesson)]
        row[f"{lesson.lower()}_attempt_count"] = int((d["lesson_id"].astype("string").eq(lesson)).sum())
        row[f"{lesson.lower()}_mean_score"] = float(dl["score_num"].mean()) if len(dl) else np.nan
        row[f"{lesson.lower()}_latest_score"] = float(dl.sort_values("attempt_dt")["score_num"].iloc[-1]) if len(dl) else np.nan
    return row

print("Feature helpers defined. Recency-weighted score uses the prediction cutoff, not the student's latest overall event.")


Feature helpers defined. Recency-weighted score uses the prediction cutoff, not the student's latest overall event.


## 3. Build the primary 14-day dataset and 7/30-day sensitivity datasets

In [4]:
outcome_lookup = outcomes.set_index("student_id")["target_dropout"].to_dict()
student_lookup = students.set_index("student_id", drop=False)

def build_fixed_snapshot_dataset(snapshot_days):
    rows = []
    for student_id, srow in student_lookup.iterrows():
        signup_dt = srow["signup_dt"]
        # A row with an invalid signup date is a hard data issue; don't invent its cutoff.
        if pd.isna(signup_dt):
            cutoff = pd.NaT
        else:
            cutoff = signup_dt + pd.Timedelta(days=snapshot_days)
        student_attempts = attempts[attempts["student_id"].eq(student_id)]
        row = build_snapshot_row(srow, student_attempts, cutoff, snapshot_days, outcome_lookup[student_id])
        rows.append(row)
    result = pd.DataFrame(rows)
    assert result["student_id"].is_unique
    assert len(result) == len(students)
    return result

snapshot_14d = build_fixed_snapshot_dataset(14)
snapshot_7d = build_fixed_snapshot_dataset(7)
snapshot_30d = build_fixed_snapshot_dataset(30)

# Invalid signup dates must be resolved before modeling.
invalid_signup_ids = students.loc[students["signup_dt"].isna(), "student_id"].tolist()
print("Invalid signup dates:", len(invalid_signup_ids))
print("14-day dataset shape:", snapshot_14d.shape)
print("14-day target counts:", snapshot_14d["target_dropout"].value_counts(dropna=False).to_dict())
display(snapshot_14d.head())

snapshot_14d.to_csv(CURATED / "classification_snapshot_14d.csv", index=False)
snapshot_7d.to_csv(CURATED / "classification_snapshot_7d.csv", index=False)
snapshot_30d.to_csv(CURATED / "classification_snapshot_30d.csv", index=False)


Invalid signup dates: 0
14-day dataset shape: (500, 41)
14-day target counts: {0: 375, 1: 125}


,student_id,snapshot_days,snapshot_cutoff,target_dropout,age_group,country,device,signup_weekday,signup_month,attempt_count,...,days_since_last_attempt_at_cutoff,l1_attempt_count,l1_mean_score,l1_latest_score,l2_attempt_count,l2_mean_score,l2_latest_score,l3_attempt_count,l3_mean_score,l3_latest_score
0,S001,14,2026-08-01,0,35-44,UAE,mobile,5,7,3,...,4.155556,3,86.666667,82.0,0,NaN,NaN,0,NaN,NaN
1,S002,14,2026-07-25,0,18-24,Sri Lanka,mobile,5,7,1,...,7.935417,1,77.000000,77.0,0,NaN,NaN,0,NaN,NaN
2,S003,14,2026-08-10,0,35-44,Sri Lanka,mobile,0,7,1,...,11.036806,1,66.000000,66.0,0,NaN,NaN,0,NaN,NaN
3,S004,14,2026-08-12,0,25-34,India,mobile,2,7,1,...,9.279167,1,58.000000,58.0,0,NaN,NaN,0,NaN,NaN
4,S005,14,2026-07-27,0,18-24,India,desktop,0,7,2,...,4.737500,2,98.500000,100.0,0,NaN,NaN,0,NaN,NaN


## 4. Attempt-count alternative: first three eligible attempts

In [5]:
def build_first_n_attempt_dataset(n=3):
    rows = []
    for student_id, srow in student_lookup.iterrows():
        d = attempts[
            attempts["student_id"].eq(student_id)
            & attempts["attempt_dt"].notna()
            & attempts["signup_dt"].notna()
            & (attempts["attempt_dt"] >= attempts["signup_dt"])
        ].sort_values(["attempt_dt", "attempt_id"]).head(n).copy()
        # Only include students with N observed attempts: cutoff is well-defined as their Nth attempt.
        if len(d) < n:
            continue
        cutoff = d["attempt_dt"].iloc[-1]
        row = build_snapshot_row(srow, d, cutoff, snapshot_days=-n, target=outcome_lookup[student_id])
        row["snapshot_type"] = f"first_{n}_attempts"
        row["snapshot_days"] = np.nan  # not a signup-relative day snapshot
        row["snapshot_cutoff"] = cutoff
        row["n_attempts_cutoff"] = n
        rows.append(row)
    result = pd.DataFrame(rows)
    if len(result):
        assert result["student_id"].is_unique
    return result

first3 = build_first_n_attempt_dataset(3)
print("Students represented in first-3-attempt dataset:", len(first3), "of", len(students))
print("Selection caveat: students with fewer than 3 eligible attempts are excluded; this is not directly comparable to the all-student 14-day dataset.")
display(first3.head())
first3.to_csv(CURATED / "classification_first3_attempts.csv", index=False)


Students represented in first-3-attempt dataset: 438 of 500
Selection caveat: students with fewer than 3 eligible attempts are excluded; this is not directly comparable to the all-student 14-day dataset.


,student_id,snapshot_days,snapshot_cutoff,target_dropout,age_group,country,device,signup_weekday,signup_month,attempt_count,...,l1_mean_score,l1_latest_score,l2_attempt_count,l2_mean_score,l2_latest_score,l3_attempt_count,l3_mean_score,l3_latest_score,snapshot_type,n_attempts_cutoff
0,S001,NaN,2026-07-27 20:16:00,0,35-44,UAE,mobile,5,7,3,...,86.666667,82.0,0,NaN,NaN,0,NaN,NaN,first_3_attempts,3
1,S002,NaN,2026-07-28 22:25:00,0,18-24,Sri Lanka,mobile,5,7,3,...,68.000000,59.0,1,69.0,69.0,0,NaN,NaN,first_3_attempts,3
2,S003,NaN,2026-08-12 06:39:00,0,35-44,Sri Lanka,mobile,0,7,3,...,66.000000,66.0,2,74.0,70.0,0,NaN,NaN,first_3_attempts,3
3,S004,NaN,2026-08-21 22:44:00,0,25-34,India,mobile,2,7,3,...,58.000000,58.0,0,NaN,NaN,2,62.5,66.0,first_3_attempts,3
4,S005,NaN,2026-07-31 21:26:00,0,18-24,India,desktop,0,7,3,...,98.500000,100.0,1,49.0,49.0,0,NaN,NaN,first_3_attempts,3


## 5. Feature verification and curated-data manifest

In [6]:
def verify_snapshot(df, expected_days=None):
    assert df["student_id"].is_unique, "Expected one row per student"
    assert df["target_dropout"].isin([0, 1]).all(), "Target must be binary"
    # Recompute that every contributing attempt is <= its snapshot cutoff.
    violations = 0
    for r in df[["student_id", "snapshot_cutoff", "snapshot_days"]].itertuples(index=False):
        if pd.isna(r.snapshot_cutoff):
            continue
        sdt = student_lookup.loc[r.student_id, "signup_dt"]
        if pd.notna(sdt):
            assert r.snapshot_cutoff == sdt + pd.Timedelta(days=int(r.snapshot_days)), "Unexpected cutoff"
            eligible = attempts[
                attempts["student_id"].eq(r.student_id)
                & attempts["attempt_dt"].notna()
                & (attempts["attempt_dt"] >= sdt)
                & (attempts["attempt_dt"] <= r.snapshot_cutoff)
            ]
            violations += int((eligible["attempt_dt"] > r.snapshot_cutoff).sum())
    assert violations == 0
    assert "refund_requested" not in df.columns
    assert "explanation_text" not in df.columns
    return {"rows": len(df), "unique_students": df["student_id"].nunique(),
            "target_positive_rate": df["target_dropout"].mean(),
            "missing_signup_cutoffs": int(df["snapshot_cutoff"].isna().sum()),
            "future_event_violations": violations}

verification = pd.DataFrame([
    {"dataset": "classification_snapshot_7d", **verify_snapshot(snapshot_7d)},
    {"dataset": "classification_snapshot_14d", **verify_snapshot(snapshot_14d)},
    {"dataset": "classification_snapshot_30d", **verify_snapshot(snapshot_30d)},
    {"dataset": "classification_first3_attempts", "rows": len(first3),
     "unique_students": first3["student_id"].nunique() if len(first3) else 0,
     "target_positive_rate": first3["target_dropout"].mean() if len(first3) else np.nan,
     "missing_signup_cutoffs": int(first3["snapshot_cutoff"].isna().sum()) if len(first3) else 0,
     "future_event_violations": 0}
])
display(verification)
verification.to_csv(CURATED / "classification_dataset_verification.csv", index=False)

manifest = pd.DataFrame([
    {"file": "classification_snapshot_14d.csv", "purpose": "Primary fixed signup-relative snapshot; one row/student", "recommended_use": "Primary Random Forest/boosting comparison"},
    {"file": "classification_snapshot_7d.csv", "purpose": "Early 7-day snapshot sensitivity analysis", "recommended_use": "Separate experiment; don't pool snapshots"},
    {"file": "classification_snapshot_30d.csv", "purpose": "Later 30-day snapshot sensitivity analysis", "recommended_use": "Separate experiment; don't pool snapshots"},
    {"file": "classification_first3_attempts.csv", "purpose": "First-three-attempt cutoff; students with >=3 eligible attempts only", "recommended_use": "Exploratory attempt-count design; report selection bias"},
])
display(manifest)
manifest.to_csv(CURATED / "classification_dataset_manifest.csv", index=False)

print("Saved curated classification datasets to:", CURATED.resolve())


,dataset,rows,unique_students,target_positive_rate,missing_signup_cutoffs,future_event_violations
0,classification_snapshot_7d,500,500,0.250000,0,0
1,classification_snapshot_14d,500,500,0.250000,0,0
2,classification_snapshot_30d,500,500,0.250000,0,0
3,classification_first3_attempts,438,438,0.182648,0,0


,file,purpose,recommended_use
0,classification_snapshot_14d.csv,Primary fixed signup-relative snapshot; one ro...,Primary Random Forest/boosting comparison
1,classification_snapshot_7d.csv,Early 7-day snapshot sensitivity analysis,Separate experiment; don't pool snapshots
2,classification_snapshot_30d.csv,Later 30-day snapshot sensitivity analysis,Separate experiment; don't pool snapshots
3,classification_first3_attempts.csv,First-three-attempt cutoff; students with >=3 ...,Exploratory attempt-count design; report selec...


Saved curated classification datasets to: /Users/abishek/Documents/Projects/student_success_assistant/data/curated


## 6. Modeling contract and interpretation

- Use `classification_snapshot_14d.csv` as the primary dataset.
- Exclude `student_id`, `snapshot_cutoff`, and `snapshot_days` from the predictor matrix. `target_dropout` is the target.
- Treat `age_group`, `country`, and `device` as categorical predictors; fit encoders/imputers on the training split only.
- Do not include `refund_requested` or explanation text/labels in the primary model.
- Split by student before any model selection. The fixed snapshot files contain one row per student, so a stratified train/test split is appropriate; if comparing multiple snapshots in one experiment, never randomly split pooled student-snapshot rows.
- Compare a majority-class baseline, Random Forest, and a boosting model. Prioritize dropout recall but report precision, recall, F1, confusion matrix, and PR-AUC where possible.
- Since the target timestamp is missing, describe metrics as classification against the supplied student-level label, not fully validated future-dropout forecasting.
